In [1]:
import pandas as pd
import scanpy as sc

/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [2]:
donor_held_out = "Donor1"
idx_given_donor = "2"

control_key = "is_control"
    
adata_train = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_train_{donor_held_out}.h5ad")


In [3]:
adata_train.uns.keys()

dict_keys(['cytokines_to_impute', 'cytokines_to_train_data', 'donor_embeddings', 'esm2_embeddings', 'hvg', 'log1p', 'split_info'])

In [24]:
import os
import pickle

embedding_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/cellflow/embeddings_from_tahoe"

with open(os.path.join(embedding_dir, "sci_cl_cell_line.pkl"), "rb") as f:
    embedding_dict1 = pickle.load(f)
        
with open(os.path.join(embedding_dir, "sci_emb.pkl"), "rb") as f:
    embedding_dict2 = pickle.load(f)
with open(os.path.join(embedding_dir, "sci_drug_concatenated_from_tahoe.pkl"), "rb") as f:
    embedding_dict3 = pickle.load(f)

In [25]:
embedding_dict1['A549_(+)-JQ1'][:10]

array([ 0.03137634,  0.04851987, -0.01447944,  0.09426983, -0.03131317,
        0.01335863, -0.02418063, -0.03780764,  0.0464353 ,  0.09233634])

In [26]:
embedding_dict2['A549_(+)-JQ1'][:10]

array([ 0.03137634,  0.04851987, -0.01447944,  0.09426983, -0.03131317,
        0.01335863, -0.02418063, -0.03780764,  0.0464353 ,  0.09233634])

In [10]:
embedding_dict3['A549_(+)-JQ1'][:10]

KeyError: 'A549_(+)-JQ1'

In [11]:
import scanpy as sc

out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/additive_model/pbmc_new_donor/closest_embedding"
donor_held_out = "Donor1"
idx_given_donor = "1"

control_key = "is_control"
    
adata_train = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_train_{donor_held_out}.h5ad")
adata_ood_perturbed  = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_ood_{donor_held_out}.h5ad")
cytokines_to_impute = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_impute"]
cytokines_to_train_data = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_train_data"]

if len(cytokines_to_train_data) == 0:
    sys.exit(0)

adata_ctrl = adata_train[adata_train.obs[control_key].to_numpy()]


with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/idcs_to_keep.pkl", "rb") as pickle_file:
    idcs_to_keep = pickle.load(pickle_file)
adata_full = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/pbmc_with_pca.h5ad")
adata_ref = adata_full[adata_full.obs_names.isin(idcs_to_keep)]
with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/degs.pkl", "rb") as pickle_file:
    deg_genes = pickle.load(pickle_file)

#with open("/lustre/groups/ml01/workspace/ot_perturbation/data/embeddings/pbmc_cytokine_mashup.pkl", "rb") as pickle_file:
#    mashup_embeddings = pickle.load(pickle_file)

donor_embeddings = adata_train.uns['donor_embeddings']


adata_ctrl_current_donor = adata_ctrl[adata_ctrl.obs["donor"]==donor_held_out]
if adata_ctrl_current_donor.n_obs > 10000:
        sc.pp.subsample(adata_ctrl_current_donor, n_obs=10000)



NameError: name 'find_closest_embedding' is not defined

In [16]:
import anndata as ad
from typing import Any, Tuple

import numpy as np
def get_train_embeddings(adata_train: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_train.obs.drop_duplicates(subset="cytokine")
    emb_vectors = {}
    for _,row in conds.iterrows():
        cyto = row["cytokine"]
        if cyto=="PBS":
            continue
        emb_vectors[cyto] = embeddings[cyto]
    return emb_vectors

def find_closest_embedding(emb_0: np.ndarray, reference_embeddings: dict[str, np.ndarray]) -> Tuple[str, ...]:
    closest_emb = None
    closest_dist = np.inf
    for ref, ref_emb in reference_embeddings.items():
        dist = np.sum((emb_0-ref_emb)**2)
        if dist < closest_dist:
            closest_dist = dist
            closest_emb = ref
    return closest_emb

In [22]:
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca


closest_donor = find_closest_embedding(donor_embeddings[donor_held_out], {k:v for k,v in donor_embeddings.items() if k!=donor_held_out})
for cytokine in cytokines_to_impute:
    adata_pred = adata_train[(adata_train.obs["cytokine"]==cytokine)&(adata_train.obs["donor"]==closest_donor)].copy()
    adata_pred.X = adata_pred.X.toarray()
    
    condition = f"{donor_held_out}_{cytokine}"
    
    project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")
    project_pca(query_adata=adata_pred, ref_adata=adata_full, obsm_key_added="X_pca")
    cond_orig = condition
    condition = condition + "_" + str(len(cytokines_to_train_data))
    donor_deg_dict = {k: v for k, v in deg_genes.items() if (k.startswith(donor_held_out) and k.endswith(f"_{cytokine}"))}
    adata_ood_true = adata_full[(adata_full.obs["donor"] == donor_held_out) & (adata_full.obs["cytokine"]==cytokine)]
    
    out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, donor_deg_dict=donor_deg_dict, adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl_current_donor)
    out["num_cytokines_in_train"] = len(cytokines_to_train_data)
    pd.DataFrame.from_dict(out, columns=[condition], orient="index").to_csv(os.path.join(out_dir, f"{idx_given_donor}_{condition}.csv"))



NameError: name 'compute_metrics' is not defined

In [23]:
adata_pred

AnnData object with n_obs × n_vars = 6326 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'condition', 'is_control'
    uns: 'cytokines_to_impute', 'cytokines_to_train_data', 'donor_embeddings', 'esm2_embeddings', 'hvg', 'log1p', 'split_info'
    obsm: 'X_pca_for_ct_transfer', 'X_pca'
    layers: 'counts'